In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv("/content/housing.csv")


In [3]:
X = data[['median_income', 'housing_median_age']]
y = data[['median_house_value']]


In [4]:
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)


In [5]:
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)


In [6]:
class LinearModel(nn.Module):
    def __init__(self):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        return self.linear(x)

In [7]:
def train_model(reg_type="none", l1_lambda=0.01, l2_lambda=0.01):
    model = LinearModel()
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01)


    print(f"REGULARIZATION TYPE : {reg_type.upper()}")
    print("Weights BEFORE update:", model.linear.weight.data)
    print("Bias BEFORE update   :", model.linear.bias.data)

    # Forward pass
    predictions = model(X_tensor)
    loss = criterion(predictions, y_tensor)

    # Add Regularization
    if reg_type == "l1":
        l1_penalty = sum(torch.sum(torch.abs(p)) for p in model.parameters())
        loss = loss + l1_lambda * l1_penalty

    elif reg_type == "l2":
        l2_penalty = sum(torch.sum(p ** 2) for p in model.parameters())
        loss = loss + l2_lambda * l2_penalty

    elif reg_type == "elastic":
        l1_penalty = sum(torch.sum(torch.abs(p)) for p in model.parameters())
        l2_penalty = sum(torch.sum(p ** 2) for p in model.parameters())
        loss = loss + l1_lambda * l1_penalty + l2_lambda * l2_penalty

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Weights AFTER update :", model.linear.weight.data)
    print("Bias AFTER update    :", model.linear.bias.data)
    print("Loss Value           :", loss.item())


# Execute All Experiments

train_model("none")
train_model("l1")
train_model("l2")
train_model("elastic")

REGULARIZATION TYPE : NONE
Weights BEFORE update: tensor([[-0.1014,  0.0066]])
Bias BEFORE update   : tensor([-0.3496])
Weights AFTER update : tensor([[-0.0856,  0.0083]])
Bias AFTER update    : tensor([-0.3426])
Loss Value           : 1.2708470821380615
REGULARIZATION TYPE : L1
Weights BEFORE update: tensor([[0.2666, 0.5990]])
Bias BEFORE update   : tensor([0.0967])
Weights AFTER update : tensor([[0.2764, 0.5897]])
Bias AFTER update    : tensor([0.0947])
Loss Value           : 0.9173895120620728
REGULARIZATION TYPE : L2
Weights BEFORE update: tensor([[-0.1734,  0.2826]])
Bias BEFORE update   : tensor([0.2812])
Weights AFTER update : tensor([[-0.1554,  0.2786]])
Bias AFTER update    : tensor([0.2755])
Loss Value           : 1.3814400434494019
REGULARIZATION TYPE : ELASTIC
Weights BEFORE update: tensor([[0.0104, 0.0342]])
Bias BEFORE update   : tensor([0.4608])
Weights AFTER update : tensor([[0.0240, 0.0355]])
Bias AFTER update    : tensor([0.4514])
Loss Value           : 1.199194550514